In [1]:
app_code = '''
import streamlit as st
import anthropic
import numpy as np
import sys
sys.path.append("/Users/mustafaabushagur")
import photonics_tools as pt

st.set_page_config(
    page_title="Optical System Design Advisor",
    page_icon="🔭",
    layout="wide"
)

st.title("Optical System Design Advisor")
st.markdown("""
*AI-powered photonics design tool built on* **Applied Photonics**  
*by Prof. Mustafa A.G. Abushagur — RIT*
""")
st.divider()

with st.sidebar:
    st.header("Setup")
    api_key = st.text_input(
        "Anthropic API Key",
        type="password",
        placeholder="sk-ant-api03-..."
    )
    st.markdown("---")
    st.header("Design Category")
    category = st.selectbox(
        "Select component type:",
        [
            "Optical Fiber",
            "Fabry-Perot Cavity",
            "Ray Optics System",
            "General Photonics Problem"
        ]
    )
    st.markdown("---")
    st.markdown("### About")
    st.markdown("""
    Describe your photonics design problem
    in plain English. The advisor will:
    - Compute relevant parameters
    - Analyze performance
    - Recommend design approaches
    - Identify tradeoffs
    """)

def compute_fiber_params(n1, n2, core_um, lambda_nm):
    NA = np.sqrt(n1**2 - n2**2)
    V  = (2 * np.pi * core_um*1e-6 / (lambda_nm*1e-9)) * NA
    acceptance = np.degrees(np.arcsin(NA))
    lambdas = np.linspace(500, 2000, 10000)
    V_curve = (2 * np.pi * core_um*1e-6 / (lambdas*1e-9)) * NA
    cutoff_idx = np.argmin(np.abs(V_curve - 2.405))
    cutoff_lambda = lambdas[cutoff_idx]
    if V < 2.405:
        mode = "single-mode"
    elif V < 3.832:
        mode = "few-mode"
    else:
        mode = "multimode"
    return {
        "NA": NA,
        "V_number": V,
        "mode": mode,
        "acceptance_angle": acceptance,
        "cutoff_lambda_nm": cutoff_lambda
    }

def compute_fabry_perot_params(R, n, d_um, lambda_nm):
    finesse   = np.pi * np.sqrt(R) / (1 - R)
    lambda_m  = lambda_nm * 1e-9
    d_m       = d_um * 1e-6
    FSR_nm    = (lambda_m**2 / (2 * n * d_m)) * 1e9
    linewidth = FSR_nm / finesse
    return {
        "finesse":   finesse,
        "FSR_nm":    FSR_nm,
        "linewidth": linewidth
    }

def get_design_recommendation(problem, computed_params, category, api_key):
    if computed_params:
        params_text = "\\n".join([f"  {k}: {v:.4f}" if isinstance(v, float) 
                                  else f"  {k}: {v}" 
                                  for k, v in computed_params.items()])
    else:
        params_text = "No parameters computed yet."
    
    prompt = f"""You are an expert photonics engineer and professor with 40 years 
of experience in optics, fiber optics, and photonic systems.

A student or engineer has described the following design problem:
{problem}

Component category: {category}

Computed parameters:
{params_text}

Please provide:
1. Assessment of the design requirements
2. Whether the computed parameters meet the requirements
3. Specific design recommendations
4. Key tradeoffs to consider
5. Potential issues or limitations
6. Suggestions for improvement

Be specific, technical, and practical. Reference relevant equations where helpful.
Render equations in LaTeX using $$ for display equations."""

    client = anthropic.Anthropic(api_key=api_key)
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

if not api_key:
    st.info("Please enter your Anthropic API key in the sidebar to begin.")
else:
    st.header(f"Design Problem — {category}")
    
    problem = st.text_area(
        "Describe your design problem:",
        placeholder="e.g. I need a single-mode fiber for 1550nm with minimum bend loss...",
        height=120
    )
    
    computed_params = {}
    
    if category == "Optical Fiber":
        st.subheader("Fiber Parameters")
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            n1 = st.number_input("Core index n1", value=1.4682, format="%.4f")
        with col2:
            n2 = st.number_input("Cladding index n2", value=1.4629, format="%.4f")
        with col3:
            core_um = st.number_input("Core radius (um)", value=4.5, format="%.2f")
        with col4:
            lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
        
        if st.button("Compute Parameters", type="secondary"):
            computed_params = compute_fiber_params(n1, n2, core_um, lambda_nm)
            
            st.subheader("Computed Parameters")
            col1, col2, col3, col4, col5 = st.columns(5)
            with col1:
                st.metric("NA", f"{computed_params['NA']:.4f}")
            with col2:
                st.metric("V-number", f"{computed_params['V_number']:.3f}")
            with col3:
                st.metric("Mode", computed_params['mode'])
            with col4:
                st.metric("Acceptance angle", f"{computed_params['acceptance_angle']:.2f}°")
            with col5:
                st.metric("Cutoff wavelength", f"{computed_params['cutoff_lambda_nm']:.1f} nm")
    
    elif category == "Fabry-Perot Cavity":
        st.subheader("Cavity Parameters")
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            R = st.number_input("Mirror reflectivity R", value=0.95, min_value=0.0, max_value=0.999, format="%.3f")
        with col2:
            n = st.number_input("Refractive index n", value=1.5, format="%.3f")
        with col3:
            d_um = st.number_input("Cavity length (um)", value=15.5, format="%.2f")
        with col4:
            lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
        
        if st.button("Compute Parameters", type="secondary"):
            computed_params = compute_fabry_perot_params(R, n, d_um, lambda_nm)
            
            st.subheader("Computed Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                st.metric("Finesse", f"{computed_params['finesse']:.2f}")
            with col2:
                st.metric("FSR", f"{computed_params['FSR_nm']:.4f} nm")
            with col3:
                st.metric("Linewidth", f"{computed_params['linewidth']:.4f} nm")
    
    elif category == "Ray Optics System":
        st.subheader("Lens System Parameters")
        col1, col2, col3 = st.columns(3)
        with col1:
            f = st.number_input("Focal length (m)", value=0.05, format="%.4f")
        with col2:
            L1 = st.number_input("Object distance (m)", value=0.1, format="%.4f")
        with col3:
            L2 = st.number_input("Image distance (m)", value=0.1, format="%.4f")
        
        if st.button("Compute Parameters", type="secondary"):
            M_system = pt.free_space(L2) @ pt.thin_lens(f) @ pt.free_space(L1)
            computed_params = {
                "A": M_system[0,0],
                "B": M_system[0,1],
                "C": M_system[1,0],
                "D": M_system[1,1],
                "imaging_condition": abs(M_system[0,1]) < 1e-6
            }
            
            st.subheader("System Matrix")
            col1, col2, col3, col4, col5 = st.columns(5)
            with col1:
                st.metric("A", f"{computed_params['A']:.4f}")
            with col2:
                st.metric("B", f"{computed_params['B']:.6f}")
            with col3:
                st.metric("C", f"{computed_params['C']:.4f}")
            with col4:
                st.metric("D", f"{computed_params['D']:.4f}")
            with col5:
                st.metric("Imaging", "Yes" if computed_params["imaging_condition"] else "No")
    
    else:
        st.info("Describe your general photonics problem above and click Get Design Recommendation.")
    
    st.markdown("---")
    
    if st.button("Get Design Recommendation", type="primary"):
        if not problem:
            st.warning("Please describe your design problem first.")
        else:
            with st.spinner("Analyzing your design problem..."):
                try:
                    recommendation = get_design_recommendation(
                        problem, computed_params, category, api_key
                    )
                    st.subheader("Design Recommendation")
                    st.markdown(recommendation)
                    
                    with open("design_log.txt", "a") as f:
                        f.write(f"Category: {category}\\n")
                        f.write(f"Problem: {problem}\\n")
                        f.write(f"Params: {computed_params}\\n")
                        f.write("---\\n")
                    
                except Exception as e:
                    st.error(f"Error: {e}")
    
    st.markdown("---")
    st.markdown("### Example design problems")
    examples = {
        "Optical Fiber": [
            "I need a single-mode fiber for 1550nm DWDM transmission over 100km",
            "Design a fiber for minimum dispersion at 1310nm",
            "I need a large core fiber for high power laser delivery"
        ],
        "Fabry-Perot Cavity": [
            "I need a narrowband filter for WDM channel selection at 1550nm",
            "Design a laser cavity with maximum mode selectivity",
            "I need a Fabry-Perot etalon for optical coherence tomography"
        ],
        "Ray Optics System": [
            "I need a collimating lens system for a laser diode",
            "Design a beam expander for a Gaussian beam",
            "I need a focusing system for fiber coupling"
        ]
    }
    
    if category in examples:
        for example in examples[category]:
            if st.button(example, key=f"ex_{example[:20]}"):
                st.session_state["problem"] = example
'''

with open("/Users/mustafaabushagur/optical_design_advisor.py", "w") as f:
    f.write(app_code)

print("Optical Design Advisor saved successfully")
print("Location: /Users/mustafaabushagur/optical_design_advisor.py")


Optical Design Advisor saved successfully
Location: /Users/mustafaabushagur/optical_design_advisor.py


In [2]:
app_code = '''
import streamlit as st
import anthropic
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("/Users/mustafaabushagur")
import photonics_tools as pt

st.set_page_config(
    page_title="Optical System Design Advisor",
    page_icon="🔭",
    layout="wide"
)

st.title("Optical System Design Advisor")
st.markdown("""
*AI-powered photonics design tool built on* **Applied Photonics**  
*by Prof. Mustafa A.G. Abushagur — RIT*
""")
st.divider()

with st.sidebar:
    st.header("Setup")
    api_key = st.text_input(
        "Anthropic API Key",
        type="password",
        placeholder="sk-ant-api03-..."
    )
    st.markdown("---")
    st.header("Design Category")
    category = st.selectbox(
        "Select component type:",
        [
            "Optical Fiber",
            "Fabry-Perot Cavity",
            "Ray Optics System",
            "Gaussian Beam",
            "WDM System",
            "Resonator Stability",
            "General Photonics Problem"
        ]
    )
    st.markdown("---")
    st.markdown("### About")
    st.markdown("""
    Describe your photonics design problem
    in plain English. The advisor will:
    - Compute relevant parameters
    - Analyze performance
    - Recommend design approaches
    - Identify tradeoffs
    """)

def compute_fiber_params(n1, n2, core_um, lambda_nm):
    NA = np.sqrt(n1**2 - n2**2)
    V  = (2 * np.pi * core_um*1e-6 / (lambda_nm*1e-9)) * NA
    acceptance = np.degrees(np.arcsin(NA))
    lambdas = np.linspace(500, 2000, 10000)
    V_curve = (2 * np.pi * core_um*1e-6 / (lambdas*1e-9)) * NA
    cutoff_idx = np.argmin(np.abs(V_curve - 2.405))
    cutoff_lambda = lambdas[cutoff_idx]
    if V < 2.405:
        mode = "single-mode"
    elif V < 3.832:
        mode = "few-mode"
    else:
        mode = "multimode"
    return {"NA": NA, "V_number": V, "mode": mode,
            "acceptance_angle": acceptance, "cutoff_lambda_nm": cutoff_lambda}

def compute_fabry_perot_params(R, n, d_um, lambda_nm):
    finesse   = np.pi * np.sqrt(R) / (1 - R)
    lambda_m  = lambda_nm * 1e-9
    d_m       = d_um * 1e-6
    FSR_nm    = (lambda_m**2 / (2 * n * d_m)) * 1e9
    linewidth = FSR_nm / finesse
    return {"finesse": finesse, "FSR_nm": FSR_nm, "linewidth": linewidth}

def compute_gaussian_beam(w0_um, lambda_nm, z_max_mm):
    w0 = w0_um * 1e-6
    lambda_m = lambda_nm * 1e-9
    z_R = np.pi * w0**2 / lambda_m
    z = np.linspace(0, z_max_mm*1e-3, 1000)
    w_z = w0 * np.sqrt(1 + (z/z_R)**2)
    divergence = np.degrees(np.arctan(lambda_m / (np.pi * w0)))
    return {
        "w0_um": w0_um,
        "rayleigh_range_mm": z_R * 1e3,
        "divergence_deg": divergence,
        "beam_at_zmax_um": w_z[-1] * 1e6
    }, z * 1e3, w_z * 1e6

def compute_wdm_system(n_channels, spacing_nm, lambda_start_nm,
                       fiber_loss_db_km, length_km, amp_spacing_km):
    channels = [lambda_start_nm + i*spacing_nm for i in range(n_channels)]
    total_loss_db = fiber_loss_db_km * amp_spacing_km
    n_amps = int(length_km / amp_spacing_km)
    total_capacity_gbps = n_channels * 10
    return {
        "n_channels": n_channels,
        "lambda_start_nm": lambda_start_nm,
        "lambda_end_nm": channels[-1],
        "span_loss_db": total_loss_db,
        "n_amplifiers": n_amps,
        "total_capacity_gbps": total_capacity_gbps,
        "channel_spacing_nm": spacing_nm
    }, channels

def compute_resonator(R1, R2, L):
    M_rt = pt.thin_lens(R1/2) @ pt.free_space(L) @ pt.thin_lens(R2/2) @ pt.free_space(L)
    m = (M_rt[0,0] + M_rt[1,1]) / 2
    stable = abs(m) <= 1
    g1 = 1 - L/R1
    g2 = 1 - L/R2
    return {
        "stability_parameter": m,
        "stable": stable,
        "g1": g1,
        "g2": g2,
        "g1_g2_product": g1*g2
    }

def get_design_recommendation(problem, computed_params, category, api_key):
    if computed_params:
        params_text = "\\n".join([
            f"  {k}: {v:.4f}" if isinstance(v, float)
            else f"  {k}: {v}"
            for k, v in computed_params.items()
        ])
    else:
        params_text = "No parameters computed yet."
    prompt = f"""You are an expert photonics engineer and professor with 40 years
of experience in optics, fiber optics, and photonic systems.

A student or engineer has described the following design problem:
{problem}

Component category: {category}

Computed parameters:
{params_text}

Please provide:
1. Assessment of the design requirements
2. Whether the computed parameters meet the requirements
3. Specific design recommendations
4. Key tradeoffs to consider
5. Potential issues or limitations
6. Suggestions for improvement

Be specific, technical, and practical.
Render equations in LaTeX using $$ for display equations."""
    client = anthropic.Anthropic(api_key=api_key)
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

if not api_key:
    st.info("Please enter your Anthropic API key in the sidebar to begin.")
else:
    st.header(f"Design Problem — {category}")
    problem = st.text_area(
        "Describe your design problem:",
        placeholder="Describe your photonics design problem in plain English...",
        height=120
    )
    computed_params = {}

    if category == "Optical Fiber":
        st.subheader("Fiber Parameters")
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            n1 = st.number_input("Core index n1", value=1.4682, format="%.4f")
        with col2:
            n2 = st.number_input("Cladding index n2", value=1.4629, format="%.4f")
        with col3:
            core_um = st.number_input("Core radius (um)", value=4.5, format="%.2f")
        with col4:
            lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
        if st.button("Compute Parameters", type="secondary"):
            computed_params = compute_fiber_params(n1, n2, core_um, lambda_nm)
            st.subheader("Computed Parameters")
            col1, col2, col3, col4, col5 = st.columns(5)
            with col1:
                st.metric("NA", f"{computed_params['NA']:.4f}")
            with col2:
                st.metric("V-number", f"{computed_params['V_number']:.3f}")
            with col3:
                st.metric("Mode", computed_params['mode'])
            with col4:
                st.metric("Acceptance angle", f"{computed_params['acceptance_angle']:.2f}°")
            with col5:
                st.metric("Cutoff wavelength", f"{computed_params['cutoff_lambda_nm']:.1f} nm")

    elif category == "Fabry-Perot Cavity":
        st.subheader("Cavity Parameters")
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            R = st.number_input("Mirror reflectivity R", value=0.95,
                                min_value=0.0, max_value=0.999, format="%.3f")
        with col2:
            n = st.number_input("Refractive index n", value=1.5, format="%.3f")
        with col3:
            d_um = st.number_input("Cavity length (um)", value=15.5, format="%.2f")
        with col4:
            lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
        if st.button("Compute Parameters", type="secondary"):
            computed_params = compute_fabry_perot_params(R, n, d_um, lambda_nm)
            st.subheader("Computed Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                st.metric("Finesse", f"{computed_params['finesse']:.2f}")
            with col2:
                st.metric("FSR", f"{computed_params['FSR_nm']:.4f} nm")
            with col3:
                st.metric("Linewidth", f"{computed_params['linewidth']:.4f} nm")

    elif category == "Ray Optics System":
        st.subheader("Lens System Parameters")
        col1, col2, col3 = st.columns(3)
        with col1:
            f = st.number_input("Focal length (m)", value=0.05, format="%.4f")
        with col2:
            L1 = st.number_input("Object distance (m)", value=0.1, format="%.4f")
        with col3:
            L2 = st.number_input("Image distance (m)", value=0.1, format="%.4f")
        if st.button("Compute Parameters", type="secondary"):
            M_system = pt.free_space(L2) @ pt.thin_lens(f) @ pt.free_space(L1)
            computed_params = {
                "A": M_system[0,0], "B": M_system[0,1],
                "C": M_system[1,0], "D": M_system[1,1],
                "imaging_condition": abs(M_system[0,1]) < 1e-6
            }
            st.subheader("System Matrix")
            col1, col2, col3, col4, col5 = st.columns(5)
            with col1:
                st.metric("A", f"{computed_params['A']:.4f}")
            with col2:
                st.metric("B", f"{computed_params['B']:.6f}")
            with col3:
                st.metric("C", f"{computed_params['C']:.4f}")
            with col4:
                st.metric("D", f"{computed_params['D']:.4f}")
            with col5:
                st.metric("Imaging", "Yes" if computed_params["imaging_condition"] else "No")

    elif category == "Gaussian Beam":
        st.subheader("Gaussian Beam Parameters")
        col1, col2, col3 = st.columns(3)
        with col1:
            w0_um = st.number_input("Beam waist w0 (um)", value=50.0, format="%.1f")
        with col2:
            lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
        with col3:
            z_max_mm = st.number_input("Propagation distance (mm)", value=100.0, format="%.1f")
        if st.button("Compute Parameters", type="secondary"):
            computed_params, z, w_z = compute_gaussian_beam(w0_um, lambda_nm, z_max_mm)
            st.subheader("Computed Parameters")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("Beam waist", f"{computed_params['w0_um']:.1f} um")
            with col2:
                st.metric("Rayleigh range", f"{computed_params['rayleigh_range_mm']:.2f} mm")
            with col3:
                st.metric("Divergence", f"{computed_params['divergence_deg']:.4f}°")
            with col4:
                st.metric(f"Beam at {z_max_mm:.0f}mm", f"{computed_params['beam_at_zmax_um']:.1f} um")
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.plot(z, w_z, color="steelblue", linewidth=2, label="Beam radius w(z)")
            ax.plot(z, -w_z, color="steelblue", linewidth=2)
            ax.fill_between(z, -w_z, w_z, alpha=0.2, color="steelblue")
            ax.axvline(x=computed_params["rayleigh_range_mm"],
                      color="coral", linestyle="--", label="Rayleigh range")
            ax.set_xlabel("Propagation distance z (mm)")
            ax.set_ylabel("Beam radius (um)")
            ax.set_title("Gaussian beam propagation")
            ax.legend()
            ax.grid(True, alpha=0.3)
            st.pyplot(fig)

    elif category == "WDM System":
        st.subheader("WDM System Parameters")
        col1, col2, col3 = st.columns(3)
        with col1:
            n_channels = st.number_input("Number of channels", value=8, min_value=1, max_value=96)
            spacing_nm = st.number_input("Channel spacing (nm)", value=0.8, format="%.2f")
        with col2:
            lambda_start = st.number_input("Start wavelength (nm)", value=1530.0, format="%.1f")
            fiber_loss = st.number_input("Fiber loss (dB/km)", value=0.2, format="%.2f")
        with col3:
            length_km = st.number_input("Link length (km)", value=100.0, format="%.1f")
            amp_spacing = st.number_input("Amplifier spacing (km)", value=80.0, format="%.1f")
        if st.button("Compute Parameters", type="secondary"):
            computed_params, channels = compute_wdm_system(
                n_channels, spacing_nm, lambda_start,
                fiber_loss, length_km, amp_spacing
            )
            st.subheader("System Summary")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("Channels", computed_params["n_channels"])
            with col2:
                st.metric("Span loss", f"{computed_params['span_loss_db']:.1f} dB")
            with col3:
                st.metric("Amplifiers needed", computed_params["n_amplifiers"])
            with col4:
                st.metric("Total capacity", f"{computed_params['total_capacity_gbps']} Gbps")
            st.subheader("Channel Wavelengths")
            cols = st.columns(min(8, n_channels))
            for i, lam in enumerate(channels[:8]):
                with cols[i % 8]:
                    st.metric(f"Ch {i+1}", f"{lam:.1f} nm")

    elif category == "Resonator Stability":
        st.subheader("Resonator Parameters")
        col1, col2, col3 = st.columns(3)
        with col1:
            R1 = st.number_input("Mirror 1 radius R1 (m)", value=0.2, format="%.4f")
        with col2:
            R2 = st.number_input("Mirror 2 radius R2 (m)", value=0.2, format="%.4f")
        with col3:
            L  = st.number_input("Cavity length L (m)", value=0.1, format="%.4f")
        if st.button("Compute Parameters", type="secondary"):
            computed_params = compute_resonator(R1, R2, L)
            st.subheader("Stability Analysis")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("Stability parameter m", f"{computed_params['stability_parameter']:.4f}")
            with col2:
                st.metric("g1", f"{computed_params['g1']:.4f}")
            with col3:
                st.metric("g2", f"{computed_params['g2']:.4f}")
            with col4:
                st.metric("g1·g2", f"{computed_params['g1_g2_product']:.4f}")
            if computed_params["stable"]:
                st.success("Resonator is STABLE — |m| ≤ 1")
            else:
                st.error("Resonator is UNSTABLE — |m| > 1")

    else:
        st.info("Describe your general photonics problem above and click Get Design Recommendation.")

    st.markdown("---")
    if st.button("Get Design Recommendation", type="primary"):
        if not problem:
            st.warning("Please describe your design problem first.")
        else:
            with st.spinner("Analyzing your design problem..."):
                try:
                    recommendation = get_design_recommendation(
                        problem, computed_params, category, api_key
                    )
                    st.subheader("Design Recommendation")
                    st.markdown(recommendation)
                    with open("design_log.txt", "a") as f:
                        f.write(f"Category: {category}\\n")
                        f.write(f"Problem: {problem}\\n")
                        f.write(f"Params: {computed_params}\\n")
                        f.write("---\\n")
                except Exception as e:
                    st.error(f"Error: {e}")

    st.markdown("---")
    st.markdown("### Example design problems")
    examples = {
        "Optical Fiber": [
            "I need a single-mode fiber for 1550nm DWDM over 100km",
            "Design a fiber for minimum dispersion at 1310nm",
        ],
        "Fabry-Perot Cavity": [
            "I need a narrowband WDM channel filter at 1550nm",
            "Design a laser cavity with maximum mode selectivity",
        ],
        "Gaussian Beam": [
            "I need to focus a 1550nm beam to a 10um spot for fiber coupling",
            "Design a beam expander for a high power laser",
        ],
        "WDM System": [
            "Plan a 100km DWDM link with 8 channels at 10Gbps each",
            "Design a metro WDM network with 40 channels",
        ],
        "Resonator Stability": [
            "Check if my laser cavity with 20cm mirrors and 10cm length is stable",
            "Design a stable hemispherical resonator",
        ],
        "Ray Optics System": [
            "I need to collimate a laser diode for fiber coupling",
            "Design a beam expander for a Gaussian beam",
        ]
    }
    if category in examples:
        for example in examples[category]:
            st.button(example, key=f"ex_{example[:20]}")
'''

with open("/Users/mustafaabushagur/optical_design_advisor.py", "w") as f:
    f.write(app_code)

print("Improved Design Advisor saved successfully")
print("New modules added:")
print("  + Gaussian Beam — propagation plot, Rayleigh range, divergence")
print("  + WDM System — channel planning, amplifier count, capacity")
print("  + Resonator Stability — stability check with g-parameters")

Improved Design Advisor saved successfully
New modules added:
  + Gaussian Beam — propagation plot, Rayleigh range, divergence
  + WDM System — channel planning, amplifier count, capacity
  + Resonator Stability — stability check with g-parameters


In [1]:
app_code = '''
import streamlit as st
import anthropic
import numpy as np
import matplotlib.pyplot as plt
import chromadb
import pypdf
import json
import sys
sys.path.append("/Users/mustafaabushagur")
import photonics_tools as pt

st.set_page_config(
    page_title="Optical System Design Advisor",
    page_icon="🔭",
    layout="wide"
)

st.title("Optical System Design Advisor")
st.markdown("""
*AI-powered photonics design tool built on* **Applied Photonics**  
*by Prof. Mustafa A.G. Abushagur — RIT*
""")
st.divider()

with st.sidebar:
    st.header("Setup")
    api_key = st.text_input(
        "Anthropic API Key",
        type="password",
        placeholder="sk-ant-api03-..."
    )
    st.markdown("---")
    st.header("Design Mode")
    mode = st.radio(
        "Choose mode:",
        ["Standard Design", "Agentic Design Agent"]
    )
    st.markdown("---")
    if mode == "Standard Design":
        category = st.selectbox(
            "Component type:",
            [
                "Optical Fiber",
                "Fabry-Perot Cavity",
                "Ray Optics System",
                "Gaussian Beam",
                "WDM System",
                "Resonator Stability",
                "General Photonics Problem"
            ]
        )
    st.markdown("---")
    st.markdown("### About")
    st.markdown("""
    **Standard mode**: compute parameters
    for a specific component type.
    
    **Agentic mode**: describe any photonics
    design goal and the AI agent autonomously
    searches your textbook, computes parameters,
    and optimizes the design.
    """)

# Tool functions
def compute_fiber(n1, n2, core_um, lambda_nm):
    NA = np.sqrt(n1**2 - n2**2)
    V  = (2 * np.pi * core_um*1e-6 / (lambda_nm*1e-9)) * NA
    acceptance = np.degrees(np.arcsin(NA))
    lambdas = np.linspace(500, 2000, 10000)
    V_curve = (2 * np.pi * core_um*1e-6 / (lambdas*1e-9)) * NA
    cutoff_idx = np.argmin(np.abs(V_curve - 2.405))
    cutoff_lambda = lambdas[cutoff_idx]
    if V < 2.405:
        mode = "single-mode"
    elif V < 3.832:
        mode = "few-mode"
    else:
        mode = "multimode"
    return {"NA": round(float(NA), 4), "V_number": round(float(V), 4),
            "mode": mode, "acceptance_angle": round(float(acceptance), 4),
            "cutoff_lambda_nm": round(float(cutoff_lambda), 1)}

def compute_fabry_perot(R, n, d_um, lambda_nm):
    finesse  = np.pi * np.sqrt(R) / (1 - R)
    lambda_m = lambda_nm * 1e-9
    d_m      = d_um * 1e-6
    FSR_nm   = (lambda_m**2 / (2 * n * d_m)) * 1e9
    linewidth = FSR_nm / finesse
    return {"finesse": round(float(finesse), 2),
            "FSR_nm": round(float(FSR_nm), 4),
            "linewidth_nm": round(float(linewidth), 4)}

def check_resonator_stability(R1, R2, L):
    g1 = 1 - L/R1
    g2 = 1 - L/R2
    product = g1 * g2
    stable = 0 <= product <= 1
    return {"g1": round(g1, 4), "g2": round(g2, 4),
            "g1_g2": round(product, 4), "stable": stable,
            "verdict": "STABLE" if stable else "UNSTABLE"}

@st.cache_resource
def load_book():
    pdf_path = "/Users/mustafaabushagur/Applied_Photonics_Book.pdf"
    reader = pypdf.PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and len(text.strip()) > 50:
            pages.append({"page_num": i + 1, "text": text.strip()})
    chunks = []
    for page in pages:
        words = page["text"].split()
        page_num = page["page_num"]
        start = 0
        while start < len(words):
            chunk_text = " ".join(words[start:start+500])
            if len(chunk_text.strip()) > 100:
                chunks.append({"chunk_id": f"page{page_num}_chunk{len(chunks)}",
                               "page_num": page_num, "text": chunk_text})
            start += 450
    chroma_client = chromadb.Client()
    try:
        collection = chroma_client.create_collection("applied_photonics")
        for i in range(0, len(chunks), 50):
            batch = chunks[i:i+50]
            collection.add(ids=[c["chunk_id"] for c in batch],
                          documents=[c["text"] for c in batch],
                          metadatas=[{"page_num": c["page_num"]} for c in batch])
    except Exception:
        collection = chroma_client.get_or_create_collection("applied_photonics")
    return collection

def search_book(query, collection):
    results = collection.query(query_texts=[query], n_results=2)
    chunks = results["documents"][0]
    pages  = [m["page_num"] for m in results["metadatas"][0]]
    response = ""
    for chunk, page in zip(chunks, pages):
        response += f"From page {page}: {chunk[:300]}...\\n\\n"
    return {"result": response, "pages": pages}

def run_agent(goal, client, collection, max_iterations=15):
    tools = [
        {"name": "compute_fiber",
         "description": "Compute NA, V-number, mode, acceptance angle, cutoff wavelength for a step-index fiber. Call multiple times to compare designs.",
         "input_schema": {"type": "object",
                         "properties": {
                             "n1": {"type": "number", "description": "Core refractive index"},
                             "n2": {"type": "number", "description": "Cladding refractive index"},
                             "core_um": {"type": "number", "description": "Core radius in microns"},
                             "lambda_nm": {"type": "number", "description": "Wavelength in nm"}},
                         "required": ["n1", "n2", "core_um", "lambda_nm"]}},
        {"name": "compute_fabry_perot",
         "description": "Compute finesse, FSR, linewidth for a Fabry-Perot cavity.",
         "input_schema": {"type": "object",
                         "properties": {
                             "R": {"type": "number", "description": "Mirror reflectivity 0-1"},
                             "n": {"type": "number", "description": "Refractive index"},
                             "d_um": {"type": "number", "description": "Cavity length in microns"},
                             "lambda_nm": {"type": "number", "description": "Wavelength in nm"}},
                         "required": ["R", "n", "d_um", "lambda_nm"]}},
        {"name": "check_resonator_stability",
         "description": "Check if a two-mirror laser resonator is stable using g-parameters.",
         "input_schema": {"type": "object",
                         "properties": {
                             "R1": {"type": "number", "description": "Mirror 1 radius of curvature in meters"},
                             "R2": {"type": "number", "description": "Mirror 2 radius of curvature in meters"},
                             "L": {"type": "number", "description": "Cavity length in meters"}},
                         "required": ["R1", "R2", "L"]}},
        {"name": "search_book",
         "description": "Search Applied Photonics textbook for theory and design guidelines.",
         "input_schema": {"type": "object",
                         "properties": {
                             "query": {"type": "string", "description": "What to search for"}},
                         "required": ["query"]}}
    ]
    
    tool_functions = {
        "compute_fiber":             lambda **k: compute_fiber(**k),
        "compute_fabry_perot":       lambda **k: compute_fabry_perot(**k),
        "check_resonator_stability": lambda **k: check_resonator_stability(**k),
        "search_book":               lambda **k: search_book(collection=collection, **k)
    }
    
    messages = [{"role": "user", "content": goal}]
    log = []
    
    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text, log
            break
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    log.append(f"Called: {block.name}({block.input})")
                    try:
                        result = tool_functions[block.name](**block.input)
                        log.append(f"Result: {result}")
                    except Exception as e:
                        result = {"error": str(e)}
                        log.append(f"Error: {e}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "user", "content": tool_results})
    
    return "Maximum iterations reached", log

def compute_fiber_params(n1, n2, core_um, lambda_nm):
    return compute_fiber(n1, n2, core_um, lambda_nm)

def compute_fabry_perot_params(R, n, d_um, lambda_nm):
    return compute_fabry_perot(R, n, d_um, lambda_nm)

def compute_gaussian_beam(w0_um, lambda_nm, z_max_mm):
    w0 = w0_um * 1e-6
    lambda_m = lambda_nm * 1e-9
    z_R = np.pi * w0**2 / lambda_m
    z = np.linspace(0, z_max_mm*1e-3, 1000)
    w_z = w0 * np.sqrt(1 + (z/z_R)**2)
    divergence = np.degrees(np.arctan(lambda_m / (np.pi * w0)))
    return {"w0_um": w0_um, "rayleigh_range_mm": z_R*1e3,
            "divergence_deg": divergence,
            "beam_at_zmax_um": w_z[-1]*1e6}, z*1e3, w_z*1e6

def compute_wdm_system(n_channels, spacing_nm, lambda_start_nm,
                       fiber_loss_db_km, length_km, amp_spacing_km):
    channels = [lambda_start_nm + i*spacing_nm for i in range(n_channels)]
    total_loss_db = fiber_loss_db_km * amp_spacing_km
    n_amps = int(length_km / amp_spacing_km)
    total_capacity_gbps = n_channels * 10
    return {"n_channels": n_channels, "lambda_start_nm": lambda_start_nm,
            "lambda_end_nm": channels[-1], "span_loss_db": total_loss_db,
            "n_amplifiers": n_amps,
            "total_capacity_gbps": total_capacity_gbps,
            "channel_spacing_nm": spacing_nm}, channels

def compute_resonator(R1, R2, L):
    result = check_resonator_stability(R1, R2, L)
    return result

def get_design_recommendation(problem, computed_params, category, api_key):
    if computed_params:
        params_text = "\\n".join([
            f"  {k}: {v:.4f}" if isinstance(v, float)
            else f"  {k}: {v}"
            for k, v in computed_params.items()
        ])
    else:
        params_text = "No parameters computed yet."
    prompt = f"""You are an expert photonics engineer and professor with 40 years
of experience. Analyze this design problem:
{problem}
Category: {category}
Computed parameters:
{params_text}
Provide: 1) Assessment 2) Whether parameters meet requirements
3) Design recommendations 4) Tradeoffs 5) Limitations 6) Improvements
Be specific and technical. Use LaTeX for equations."""
    client_temp = anthropic.Anthropic(api_key=api_key)
    message = client_temp.messages.create(
        model="claude-sonnet-4-6", max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

if not api_key:
    st.info("Please enter your Anthropic API key in the sidebar to begin.")
else:
    claude_client = anthropic.Anthropic(api_key=api_key)
    
    if mode == "Agentic Design Agent":
        st.header("Agentic Photonics Design Agent")
        st.markdown("""
        Describe any photonics design goal. The agent will autonomously:
        - Search your Applied Photonics textbook for theory
        - Compute and optimize parameters
        - Verify all requirements are met
        - Deliver a complete specification with textbook references
        """)
        
        with st.spinner("Loading Applied Photonics book..."):
            collection = load_book()
        st.success(f"Book loaded — {collection.count()} passages ready")
        
        goal = st.text_area(
            "Describe your design goal:",
            placeholder="e.g. Design a complete fiber laser system for OCT at 1310nm...",
            height=150
        )
        
        st.markdown("### Example goals")
        example_goals = [
            "Design a single-mode fiber optimized for 1550nm DWDM with NA > 0.12",
            "Find a stable laser resonator cavity for HeNe laser at 632nm",
            "Design a Fabry-Perot filter with finesse > 50 for WDM channel selection",
            "Design a complete OCT system at 1310nm with fiber, cavity, and filter",
            "Optimize a fiber design that is single-mode at 1550nm but few-mode at 800nm"
        ]
        for example in example_goals:
            if st.button(example, key=f"ag_{example[:20]}"):
                goal = example
        
        if st.button("Run Design Agent", type="primary") and goal:
            with st.spinner("Agent working — searching book and computing parameters..."):
                answer, log = run_agent(goal, claude_client, collection)
            
            st.subheader("Design Specification")
            st.markdown(answer)
            
            with st.expander("View agent tool calls"):
                for entry in log:
                    st.text(entry)
    
    else:
        st.header(f"Standard Design — {category}")
        problem = st.text_area(
            "Describe your design problem:",
            placeholder="Describe your photonics design problem...",
            height=120
        )
        computed_params = {}
        
        if category == "Optical Fiber":
            st.subheader("Fiber Parameters")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                n1 = st.number_input("Core index n1", value=1.4682, format="%.4f")
            with col2:
                n2 = st.number_input("Cladding index n2", value=1.4629, format="%.4f")
            with col3:
                core_um = st.number_input("Core radius (um)", value=4.5, format="%.2f")
            with col4:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = compute_fiber_params(n1, n2, core_um, lambda_nm)
                st.subheader("Computed Parameters")
                col1, col2, col3, col4, col5 = st.columns(5)
                with col1:
                    st.metric("NA", f"{computed_params['NA']:.4f}")
                with col2:
                    st.metric("V-number", f"{computed_params['V_number']:.3f}")
                with col3:
                    st.metric("Mode", computed_params['mode'])
                with col4:
                    st.metric("Acceptance angle", f"{computed_params['acceptance_angle']:.2f}°")
                with col5:
                    st.metric("Cutoff wavelength", f"{computed_params['cutoff_lambda_nm']:.1f} nm")

        elif category == "Fabry-Perot Cavity":
            st.subheader("Cavity Parameters")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                R = st.number_input("Mirror reflectivity R", value=0.95,
                                    min_value=0.0, max_value=0.999, format="%.3f")
            with col2:
                n = st.number_input("Refractive index n", value=1.5, format="%.3f")
            with col3:
                d_um = st.number_input("Cavity length (um)", value=15.5, format="%.2f")
            with col4:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = compute_fabry_perot_params(R, n, d_um, lambda_nm)
                st.subheader("Computed Parameters")
                col1, col2, col3 = st.columns(3)
                with col1:
                    st.metric("Finesse", f"{computed_params['finesse']:.2f}")
                with col2:
                    st.metric("FSR", f"{computed_params['FSR_nm']:.4f} nm")
                with col3:
                    st.metric("Linewidth", f"{computed_params['linewidth_nm']:.4f} nm")

        elif category == "Ray Optics System":
            st.subheader("Lens System Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                f = st.number_input("Focal length (m)", value=0.05, format="%.4f")
            with col2:
                L1 = st.number_input("Object distance (m)", value=0.1, format="%.4f")
            with col3:
                L2 = st.number_input("Image distance (m)", value=0.1, format="%.4f")
            if st.button("Compute Parameters", type="secondary"):
                M_system = pt.free_space(L2) @ pt.thin_lens(f) @ pt.free_space(L1)
                computed_params = {"A": M_system[0,0], "B": M_system[0,1],
                                  "C": M_system[1,0], "D": M_system[1,1],
                                  "imaging_condition": abs(M_system[0,1]) < 1e-6}
                st.subheader("System Matrix")
                col1, col2, col3, col4, col5 = st.columns(5)
                with col1:
                    st.metric("A", f"{computed_params['A']:.4f}")
                with col2:
                    st.metric("B", f"{computed_params['B']:.6f}")
                with col3:
                    st.metric("C", f"{computed_params['C']:.4f}")
                with col4:
                    st.metric("D", f"{computed_params['D']:.4f}")
                with col5:
                    st.metric("Imaging", "Yes" if computed_params["imaging_condition"] else "No")

        elif category == "Gaussian Beam":
            st.subheader("Gaussian Beam Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                w0_um = st.number_input("Beam waist w0 (um)", value=50.0, format="%.1f")
            with col2:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            with col3:
                z_max_mm = st.number_input("Propagation distance (mm)", value=100.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params, z, w_z = compute_gaussian_beam(w0_um, lambda_nm, z_max_mm)
                st.subheader("Computed Parameters")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("Beam waist", f"{computed_params['w0_um']:.1f} um")
                with col2:
                    st.metric("Rayleigh range", f"{computed_params['rayleigh_range_mm']:.2f} mm")
                with col3:
                    st.metric("Divergence", f"{computed_params['divergence_deg']:.4f}°")
                with col4:
                    st.metric(f"Beam at {z_max_mm:.0f}mm", f"{computed_params['beam_at_zmax_um']:.1f} um")
                fig, ax = plt.subplots(figsize=(8, 3))
                ax.plot(z, w_z, color="steelblue", linewidth=2)
                ax.plot(z, -w_z, color="steelblue", linewidth=2)
                ax.fill_between(z, -w_z, w_z, alpha=0.2, color="steelblue")
                ax.axvline(x=computed_params["rayleigh_range_mm"],
                          color="coral", linestyle="--", label="Rayleigh range")
                ax.set_xlabel("z (mm)")
                ax.set_ylabel("Beam radius (um)")
                ax.set_title("Gaussian beam propagation")
                ax.legend()
                ax.grid(True, alpha=0.3)
                st.pyplot(fig)

        elif category == "WDM System":
            st.subheader("WDM System Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                n_channels = st.number_input("Number of channels", value=8,
                                             min_value=1, max_value=96)
                spacing_nm = st.number_input("Channel spacing (nm)", value=0.8, format="%.2f")
            with col2:
                lambda_start = st.number_input("Start wavelength (nm)", value=1530.0, format="%.1f")
                fiber_loss = st.number_input("Fiber loss (dB/km)", value=0.2, format="%.2f")
            with col3:
                length_km = st.number_input("Link length (km)", value=100.0, format="%.1f")
                amp_spacing = st.number_input("Amplifier spacing (km)", value=80.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params, channels = compute_wdm_system(
                    n_channels, spacing_nm, lambda_start,
                    fiber_loss, length_km, amp_spacing)
                st.subheader("System Summary")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("Channels", computed_params["n_channels"])
                with col2:
                    st.metric("Span loss", f"{computed_params['span_loss_db']:.1f} dB")
                with col3:
                    st.metric("Amplifiers", computed_params["n_amplifiers"])
                with col4:
                    st.metric("Capacity", f"{computed_params['total_capacity_gbps']} Gbps")
                st.subheader("Channel Wavelengths")
                cols = st.columns(min(8, n_channels))
                for i, lam in enumerate(channels[:8]):
                    with cols[i % 8]:
                        st.metric(f"Ch {i+1}", f"{lam:.1f} nm")

        elif category == "Resonator Stability":
            st.subheader("Resonator Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                R1 = st.number_input("Mirror 1 radius R1 (m)", value=0.2, format="%.4f")
            with col2:
                R2 = st.number_input("Mirror 2 radius R2 (m)", value=0.2, format="%.4f")
            with col3:
                L  = st.number_input("Cavity length L (m)", value=0.1, format="%.4f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = compute_resonator(R1, R2, L)
                st.subheader("Stability Analysis")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("m", f"{computed_params['g1_g2']:.4f}")
                with col2:
                    st.metric("g1", f"{computed_params['g1']:.4f}")
                with col3:
                    st.metric("g2", f"{computed_params['g2']:.4f}")
                with col4:
                    st.metric("g1·g2", f"{computed_params['g1_g2']:.4f}")
                if computed_params["stable"]:
                    st.success("Resonator is STABLE")
                else:
                    st.error("Resonator is UNSTABLE")
        
        else:
            st.info("Describe your problem above and click Get Design Recommendation.")
        
        st.markdown("---")
        if st.button("Get Design Recommendation", type="primary"):
            if not problem:
                st.warning("Please describe your design problem first.")
            else:
                with st.spinner("Analyzing your design..."):
                    try:
                        recommendation = get_design_recommendation(
                            problem, computed_params, category, api_key)
                        st.subheader("Design Recommendation")
                        st.markdown(recommendation)
                    except Exception as e:
                        st.error(f"Error: {e}")
'''

with open("/Users/mustafaabushagur/optical_design_advisor.py", "w") as f:
    f.write(app_code)

print("Upgraded Design Advisor saved successfully")
print("New feature: Agentic Design Agent mode")
print("Switch between Standard and Agentic in the sidebar")

Upgraded Design Advisor saved successfully
New feature: Agentic Design Agent mode
Switch between Standard and Agentic in the sidebar


In [1]:
app_code = '''
import streamlit as st
import anthropic
import numpy as np
import matplotlib.pyplot as plt
import chromadb
import pypdf
import json
import time
import sys
sys.path.append("/Users/mustafaabushagur")
import photonics_tools as pt

st.set_page_config(
    page_title="Optical System Design Advisor",
    page_icon="🔭",
    layout="wide"
)

st.title("Optical System Design Advisor")
st.markdown("""
*AI-powered photonics design tool built on* **Applied Photonics**  
*by Prof. Mustafa A.G. Abushagur — RIT*
""")
st.divider()

with st.sidebar:
    st.header("Setup")
    api_key = st.text_input(
        "Anthropic API Key",
        type="password",
        placeholder="sk-ant-api03-..."
    )
    st.markdown("---")
    st.header("Design Mode")
    mode = st.radio(
        "Choose mode:",
        [
            "Standard Design",
            "Agentic Design Agent",
            "APRA — Research Assistant"
        ]
    )
    st.markdown("---")
    if mode == "Standard Design":
        category = st.selectbox(
            "Component type:",
            [
                "Optical Fiber",
                "Fabry-Perot Cavity",
                "Ray Optics System",
                "Gaussian Beam",
                "WDM System",
                "Resonator Stability",
                "General Photonics Problem"
            ]
        )
    st.markdown("---")
    st.markdown("### Modes")
    st.markdown("""
    **Standard** — compute parameters
    for a specific component.
    
    **Agentic** — describe a design goal,
    Claude autonomously computes and optimizes.
    
    **APRA** — full research assistant.
    Searches your textbook, computes all
    parameters, and writes a complete
    engineering report automatically.
    """)

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

def compute_fiber(n1, n2, core_um, lambda_nm):
    NA = np.sqrt(n1**2 - n2**2)
    V  = (2 * np.pi * core_um*1e-6 / (lambda_nm*1e-9)) * NA
    acceptance = np.degrees(np.arcsin(NA))
    lambdas = np.linspace(500, 2000, 10000)
    V_curve = (2 * np.pi * core_um*1e-6 / (lambdas*1e-9)) * NA
    cutoff_idx = np.argmin(np.abs(V_curve - 2.405))
    cutoff_lambda = lambdas[cutoff_idx]
    if V < 2.405:
        mode = "single-mode"
    elif V < 3.832:
        mode = "few-mode"
    else:
        mode = "multimode"
    return {"NA": round(float(NA), 4), "V_number": round(float(V), 4),
            "mode": mode, "acceptance_angle": round(float(acceptance), 4),
            "cutoff_lambda_nm": round(float(cutoff_lambda), 1)}

def compute_fabry_perot(R, n, d_um, lambda_nm):
    finesse  = np.pi * np.sqrt(R) / (1 - R)
    lambda_m = lambda_nm * 1e-9
    d_m      = d_um * 1e-6
    FSR_nm   = (lambda_m**2 / (2 * n * d_m)) * 1e9
    linewidth = FSR_nm / finesse
    return {"finesse": round(float(finesse), 2),
            "FSR_nm": round(float(FSR_nm), 4),
            "linewidth_nm": round(float(linewidth), 4)}

def check_resonator_stability(R1, R2, L):
    g1 = 1 - L/R1
    g2 = 1 - L/R2
    product = g1 * g2
    stable = bool(0 <= product <= 1)
    return {"g1": round(g1, 4), "g2": round(g2, 4),
            "g1_g2": round(product, 4), "stable": stable,
            "verdict": "STABLE" if stable else "UNSTABLE"}

def plan_wdm_channels(capacity_gbps, bits_per_symbol,
                      baud_rate_gbps, channel_spacing_nm, lambda_start_nm):
    capacity_per_channel = baud_rate_gbps * bits_per_symbol
    n_channels = int(np.ceil(capacity_gbps / capacity_per_channel))
    channels   = [lambda_start_nm + i*channel_spacing_nm for i in range(n_channels)]
    total_bw   = (n_channels - 1) * channel_spacing_nm
    return {"n_channels": n_channels,
            "capacity_per_ch_gbps": capacity_per_channel,
            "total_capacity_gbps": n_channels * capacity_per_channel,
            "lambda_start_nm": lambda_start_nm,
            "lambda_end_nm": channels[-1],
            "total_bandwidth_nm": round(total_bw, 2),
            "channel_list_nm": channels[:8]}

def compute_amplifier_placement(length_km, max_span_km,
                                fiber_loss_db_km, amp_gain_db):
    n_amps      = int(np.ceil(length_km / max_span_km))
    span_length = length_km / n_amps
    span_loss   = fiber_loss_db_km * span_length
    positions   = [round(span_length * (i+1), 1) for i in range(n_amps)]
    return {"n_amplifiers": n_amps,
            "span_length_km": round(span_length, 1),
            "span_loss_db": round(span_loss, 2),
            "amp_gain_db": amp_gain_db,
            "amp_positions_km": positions,
            "gain_margin_db": round(amp_gain_db - span_loss, 2)}

def compute_link_budget(length_km, fiber_loss_db_km, connector_loss_db,
                        n_connectors, amp_gain_db, n_amps):
    fiber_loss     = fiber_loss_db_km * length_km
    connector_loss = connector_loss_db * n_connectors
    total_loss     = fiber_loss + connector_loss
    total_gain     = amp_gain_db * n_amps
    net_budget     = total_gain - total_loss
    return {"fiber_loss_db": round(fiber_loss, 2),
            "connector_loss_db": round(connector_loss, 2),
            "total_loss_db": round(total_loss, 2),
            "total_gain_db": round(total_gain, 2),
            "net_budget_db": round(net_budget, 2),
            "feasible": bool(net_budget >= 0)}

def estimate_osnr(launch_power_dbm, span_loss_db,
                  n_amps, noise_figure_db, bandwidth_ghz=12.5):
    h  = 6.626e-34
    nu = 193.4e12
    signal_power = 10**((launch_power_dbm - span_loss_db)/10) * 1e-3
    nf_linear    = 10**(noise_figure_db/10)
    noise_power  = n_amps * h * nu * nf_linear * bandwidth_ghz * 1e9
    osnr_linear  = signal_power / noise_power
    osnr_db      = 10 * np.log10(osnr_linear)
    return {"osnr_db": round(float(osnr_db), 2),
            "osnr_adequate": bool(osnr_db >= 15),
            "margin_db": round(float(osnr_db - 15), 2)}

@st.cache_resource
def load_book():
    pdf_path = "/Users/mustafaabushagur/Applied_Photonics_Book.pdf"
    reader = pypdf.PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and len(text.strip()) > 50:
            pages.append({"page_num": i + 1, "text": text.strip()})
    chunks = []
    for page in pages:
        words = page["text"].split()
        page_num = page["page_num"]
        start = 0
        while start < len(words):
            chunk_text = " ".join(words[start:start+500])
            if len(chunk_text.strip()) > 100:
                chunks.append({"chunk_id": f"page{page_num}_chunk{len(chunks)}",
                               "page_num": page_num, "text": chunk_text})
            start += 450
    chroma_client = chromadb.Client()
    try:
        collection = chroma_client.create_collection("applied_photonics")
        for i in range(0, len(chunks), 50):
            batch = chunks[i:i+50]
            collection.add(ids=[c["chunk_id"] for c in batch],
                          documents=[c["text"] for c in batch],
                          metadatas=[{"page_num": c["page_num"]} for c in batch])
    except Exception:
        collection = chroma_client.get_or_create_collection("applied_photonics")
    return collection

def search_book(query, collection):
    results = collection.query(query_texts=[query], n_results=2)
    chunks = results["documents"][0]
    pages  = [m["page_num"] for m in results["metadatas"][0]]
    response = ""
    for chunk, page in zip(chunks, pages):
        response += f"From page {page}: {chunk[:300]}...\\n\\n"
    return {"result": response, "pages": pages}

def get_all_tools():
    return [
        {"name": "compute_fiber",
         "description": "Compute NA, V-number, mode, acceptance angle, cutoff wavelength for a step-index fiber.",
         "input_schema": {"type": "object",
                         "properties": {
                             "n1": {"type": "number"}, "n2": {"type": "number"},
                             "core_um": {"type": "number"}, "lambda_nm": {"type": "number"}},
                         "required": ["n1", "n2", "core_um", "lambda_nm"]}},
        {"name": "compute_fabry_perot",
         "description": "Compute finesse, FSR, linewidth for a Fabry-Perot cavity.",
         "input_schema": {"type": "object",
                         "properties": {
                             "R": {"type": "number"}, "n": {"type": "number"},
                             "d_um": {"type": "number"}, "lambda_nm": {"type": "number"}},
                         "required": ["R", "n", "d_um", "lambda_nm"]}},
        {"name": "check_resonator_stability",
         "description": "Check if a two-mirror laser resonator is stable.",
         "input_schema": {"type": "object",
                         "properties": {
                             "R1": {"type": "number"}, "R2": {"type": "number"},
                             "L": {"type": "number"}},
                         "required": ["R1", "R2", "L"]}},
        {"name": "plan_wdm_channels",
         "description": "Plan WDM channel allocation for required capacity.",
         "input_schema": {"type": "object",
                         "properties": {
                             "capacity_gbps": {"type": "number"},
                             "bits_per_symbol": {"type": "number"},
                             "baud_rate_gbps": {"type": "number"},
                             "channel_spacing_nm": {"type": "number"},
                             "lambda_start_nm": {"type": "number"}},
                         "required": ["capacity_gbps", "bits_per_symbol",
                                     "baud_rate_gbps", "channel_spacing_nm", "lambda_start_nm"]}},
        {"name": "compute_amplifier_placement",
         "description": "Compute optimal EDFA amplifier placement.",
         "input_schema": {"type": "object",
                         "properties": {
                             "length_km": {"type": "number"},
                             "max_span_km": {"type": "number"},
                             "fiber_loss_db_km": {"type": "number"},
                             "amp_gain_db": {"type": "number"}},
                         "required": ["length_km", "max_span_km",
                                     "fiber_loss_db_km", "amp_gain_db"]}},
        {"name": "compute_link_budget",
         "description": "Compute power budget for a fiber link.",
         "input_schema": {"type": "object",
                         "properties": {
                             "length_km": {"type": "number"},
                             "fiber_loss_db_km": {"type": "number"},
                             "connector_loss_db": {"type": "number"},
                             "n_connectors": {"type": "number"},
                             "amp_gain_db": {"type": "number"},
                             "n_amps": {"type": "number"}},
                         "required": ["length_km", "fiber_loss_db_km",
                                     "connector_loss_db", "n_connectors",
                                     "amp_gain_db", "n_amps"]}},
        {"name": "estimate_osnr",
         "description": "Estimate OSNR. Must exceed 15dB for reliable transmission.",
         "input_schema": {"type": "object",
                         "properties": {
                             "launch_power_dbm": {"type": "number"},
                             "span_loss_db": {"type": "number"},
                             "n_amps": {"type": "number"},
                             "noise_figure_db": {"type": "number"}},
                         "required": ["launch_power_dbm", "span_loss_db",
                                     "n_amps", "noise_figure_db"]}},
        {"name": "search_book",
         "description": "Search Applied Photonics textbook for theory and design guidelines. Always search before making design recommendations.",
         "input_schema": {"type": "object",
                         "properties": {
                             "query": {"type": "string"}},
                         "required": ["query"]}}
    ]

def run_apra_web(question, claude_client, collection, max_iterations=20):
    system_prompt = """You are APRA — Autonomous Photonics Research Assistant,
built on Applied Photonics by Professor Mustafa A.G. Abushagur (Springer 2025).

Always search the textbook first, then compute parameters, then write a report.

Your report must include:
- Executive Summary
- Theoretical Background (with page references)
- System Design (parameters in tables)
- Performance Verification
- Bill of Materials
- Conclusions

Keep the report concise. Use LaTeX for equations.
End with: *Report by APRA · Applied Photonics, Prof. M.A.G. Abushagur, Springer 2025*"""

    tools = get_all_tools()
    tool_functions = {
        "compute_fiber":               lambda **k: compute_fiber(**k),
        "compute_fabry_perot":         lambda **k: compute_fabry_perot(**k),
        "check_resonator_stability":   lambda **k: check_resonator_stability(**k),
        "plan_wdm_channels":           lambda **k: plan_wdm_channels(**k),
        "compute_amplifier_placement": lambda **k: compute_amplifier_placement(**k),
        "compute_link_budget":         lambda **k: compute_link_budget(**k),
        "estimate_osnr":               lambda **k: estimate_osnr(**k),
        "search_book":                 lambda **k: search_book(collection=collection, **k)
    }
    
    messages  = [{"role": "user", "content": question}]
    iteration = 0
    tool_log  = []
    
    while iteration < max_iterations:
        iteration += 1
        
        for attempt in range(3):
            try:
                response = claude_client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=8192,
                    system=system_prompt,
                    tools=tools,
                    messages=messages
                )
                break
            except Exception as e:
                if "overloaded" in str(e).lower():
                    time.sleep(30)
                else:
                    raise e
        
        if response.stop_reason == "end_turn":
            report = ""
            for block in response.content:
                if hasattr(block, "text"):
                    report += block.text
            return report, tool_log, iteration
        
        if response.stop_reason == "max_tokens":
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", 
                           "content": "Please continue and complete the report."})
            continue
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_log.append(f"[{iteration}] {block.name}")
                    try:
                        result = tool_functions[block.name](**block.input)
                        result = make_json_serializable(result)
                    except Exception as e:
                        result = {"error": str(e)}
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "user", "content": tool_results})
    
    return "Maximum iterations reached", tool_log, iteration

def run_agent_web(goal, claude_client, collection, max_iterations=15):
    tools = get_all_tools()
    tool_functions = {
        "compute_fiber":               lambda **k: compute_fiber(**k),
        "compute_fabry_perot":         lambda **k: compute_fabry_perot(**k),
        "check_resonator_stability":   lambda **k: check_resonator_stability(**k),
        "plan_wdm_channels":           lambda **k: plan_wdm_channels(**k),
        "compute_amplifier_placement": lambda **k: compute_amplifier_placement(**k),
        "compute_link_budget":         lambda **k: compute_link_budget(**k),
        "estimate_osnr":               lambda **k: estimate_osnr(**k),
        "search_book":                 lambda **k: search_book(collection=collection, **k)
    }
    
    messages  = [{"role": "user", "content": goal}]
    iteration = 0
    log       = []
    
    while iteration < max_iterations:
        iteration += 1
        for attempt in range(3):
            try:
                response = claude_client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=4096,
                    tools=tools,
                    messages=messages
                )
                break
            except Exception as e:
                if "overloaded" in str(e).lower():
                    time.sleep(30)
                else:
                    raise e
        
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text, log
            break
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    log.append(f"Called: {block.name}({block.input})")
                    try:
                        result = tool_functions[block.name](**block.input)
                        result = make_json_serializable(result)
                        log.append(f"Result: {result}")
                    except Exception as e:
                        result = {"error": str(e)}
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "user", "content": tool_results})
    
    return "Maximum iterations reached", log

def get_design_recommendation(problem, computed_params, category, api_key):
    if computed_params:
        params_text = "\\n".join([
            f"  {k}: {v:.4f}" if isinstance(v, float)
            else f"  {k}: {v}"
            for k, v in computed_params.items()
        ])
    else:
        params_text = "No parameters computed yet."
    prompt = f"""You are an expert photonics engineer. Analyze:
Problem: {problem}
Category: {category}
Parameters: {params_text}
Provide: Assessment, recommendations, tradeoffs, improvements.
Use LaTeX for equations."""
    client_temp = anthropic.Anthropic(api_key=api_key)
    message = client_temp.messages.create(
        model="claude-sonnet-4-6", max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

if not api_key:
    st.info("Please enter your Anthropic API key in the sidebar to begin.")
else:
    claude_client = anthropic.Anthropic(api_key=api_key)
    
    if mode == "APRA — Research Assistant":
        st.header("APRA — Autonomous Photonics Research Assistant")
        st.markdown("""
        Describe any photonics research question or design challenge.
        APRA will autonomously search your Applied Photonics textbook,
        compute all parameters, verify requirements, and write a complete
        engineering report — with textbook page references.
        """)
        
        with st.spinner("Loading Applied Photonics textbook..."):
            collection = load_book()
        st.success(f"Textbook loaded — {collection.count()} passages indexed")
        
        question = st.text_area(
            "Research question or design challenge:",
            placeholder="e.g. Design a fiber optic gyroscope for aviation navigation at 1550nm...",
            height=150
        )
        
        st.markdown("### Example research questions")
        examples = [
            "Design a fiber optic gyroscope for aviation at 1550nm",
            "Design a DTS system for 50km pipeline monitoring",
            "Design a complete OCT system at 1310nm",
            "Design a 400 Gbps DWDM link from NYC to London",
            "Design a single-mode fiber laser at 1550nm",
            "Design a WDM passive optical network for 64 subscribers"
        ]
        cols = st.columns(3)
        for i, example in enumerate(examples):
            with cols[i % 3]:
                if st.button(example, key=f"apra_{i}"):
                    question = example
        
        col1, col2 = st.columns([1, 5])
        with col1:
            run_button = st.button("Run APRA", type="primary")
        
        if run_button and question:
            with st.spinner("APRA working — searching textbook and computing parameters..."):
                try:
                    report, tool_log, iterations = run_apra_web(
                        question, claude_client, collection
                    )
                    st.subheader("Engineering Report")
                    st.markdown(report)
                    st.markdown("---")
                    col1, col2 = st.columns(2)
                    with col1:
                        st.metric("Total iterations", iterations)
                    with col2:
                        st.metric("Tool calls", len(tool_log))
                    with st.expander("View tool call log"):
                        for entry in tool_log:
                            st.text(entry)
                    with open("apra_reports.txt", "a") as f:
                        f.write(f"Question: {question}\\n")
                        f.write(f"Report:\\n{report}\\n")
                        f.write("=" * 60 + "\\n")
                except Exception as e:
                    st.error(f"Error: {e}")
    
    elif mode == "Agentic Design Agent":
        st.header("Agentic Photonics Design Agent")
        st.markdown("""
        Describe any photonics design goal. The agent will autonomously
        search your textbook, compute and optimize parameters, and deliver
        a complete specification with textbook references.
        """)
        
        with st.spinner("Loading Applied Photonics textbook..."):
            collection = load_book()
        st.success(f"Book loaded — {collection.count()} passages ready")
        
        goal = st.text_area(
            "Describe your design goal:",
            placeholder="e.g. Design a complete fiber laser system for OCT at 1310nm...",
            height=150
        )
        
        st.markdown("### Example goals")
        examples = [
            "Design a single-mode fiber for 1550nm DWDM with NA > 0.12",
            "Find a stable laser resonator for HeNe at 632nm",
            "Design a Fabry-Perot filter with finesse > 50",
            "Plan a 400 Gbps DWDM link over 560km",
            "Optimize a fiber for single-mode at 1550nm and few-mode at 800nm"
        ]
        for example in examples:
            if st.button(example, key=f"ag_{example[:20]}"):
                goal = example
        
        if st.button("Run Design Agent", type="primary") and goal:
            with st.spinner("Agent working..."):
                try:
                    answer, log = run_agent_web(goal, claude_client, collection)
                    st.subheader("Design Specification")
                    st.markdown(answer)
                    with st.expander("View agent tool calls"):
                        for entry in log:
                            st.text(entry)
                except Exception as e:
                    st.error(f"Error: {e}")
    
    else:
        st.header(f"Standard Design — {category}")
        problem = st.text_area(
            "Describe your design problem:",
            placeholder="Describe your photonics design problem...",
            height=120
        )
        computed_params = {}
        
        if category == "Optical Fiber":
            st.subheader("Fiber Parameters")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                n1 = st.number_input("Core index n1", value=1.4682, format="%.4f")
            with col2:
                n2 = st.number_input("Cladding index n2", value=1.4629, format="%.4f")
            with col3:
                core_um = st.number_input("Core radius (um)", value=4.5, format="%.2f")
            with col4:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = compute_fiber(n1, n2, core_um, lambda_nm)
                st.subheader("Computed Parameters")
                col1, col2, col3, col4, col5 = st.columns(5)
                with col1:
                    st.metric("NA", f"{computed_params['NA']:.4f}")
                with col2:
                    st.metric("V-number", f"{computed_params['V_number']:.3f}")
                with col3:
                    st.metric("Mode", computed_params['mode'])
                with col4:
                    st.metric("Acceptance angle", f"{computed_params['acceptance_angle']:.2f}°")
                with col5:
                    st.metric("Cutoff wavelength", f"{computed_params['cutoff_lambda_nm']:.1f} nm")

        elif category == "Fabry-Perot Cavity":
            st.subheader("Cavity Parameters")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                R = st.number_input("Mirror reflectivity R", value=0.95,
                                    min_value=0.0, max_value=0.999, format="%.3f")
            with col2:
                n = st.number_input("Refractive index n", value=1.5, format="%.3f")
            with col3:
                d_um = st.number_input("Cavity length (um)", value=15.5, format="%.2f")
            with col4:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = compute_fabry_perot(R, n, d_um, lambda_nm)
                st.subheader("Computed Parameters")
                col1, col2, col3 = st.columns(3)
                with col1:
                    st.metric("Finesse", f"{computed_params['finesse']:.2f}")
                with col2:
                    st.metric("FSR", f"{computed_params['FSR_nm']:.4f} nm")
                with col3:
                    st.metric("Linewidth", f"{computed_params['linewidth_nm']:.4f} nm")

        elif category == "Ray Optics System":
            st.subheader("Lens System Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                f = st.number_input("Focal length (m)", value=0.05, format="%.4f")
            with col2:
                L1 = st.number_input("Object distance (m)", value=0.1, format="%.4f")
            with col3:
                L2 = st.number_input("Image distance (m)", value=0.1, format="%.4f")
            if st.button("Compute Parameters", type="secondary"):
                M_system = pt.free_space(L2) @ pt.thin_lens(f) @ pt.free_space(L1)
                computed_params = {
                    "A": float(M_system[0,0]), "B": float(M_system[0,1]),
                    "C": float(M_system[1,0]), "D": float(M_system[1,1]),
                    "imaging_condition": bool(abs(M_system[0,1]) < 1e-6)
                }
                st.subheader("System Matrix")
                col1, col2, col3, col4, col5 = st.columns(5)
                with col1:
                    st.metric("A", f"{computed_params['A']:.4f}")
                with col2:
                    st.metric("B", f"{computed_params['B']:.6f}")
                with col3:
                    st.metric("C", f"{computed_params['C']:.4f}")
                with col4:
                    st.metric("D", f"{computed_params['D']:.4f}")
                with col5:
                    st.metric("Imaging", "Yes" if computed_params["imaging_condition"] else "No")

        elif category == "Gaussian Beam":
            st.subheader("Gaussian Beam Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                w0_um = st.number_input("Beam waist w0 (um)", value=50.0, format="%.1f")
            with col2:
                lambda_nm = st.number_input("Wavelength (nm)", value=1550.0, format="%.1f")
            with col3:
                z_max_mm = st.number_input("Propagation distance (mm)", value=100.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                w0 = w0_um * 1e-6
                lambda_m = lambda_nm * 1e-9
                z_R = np.pi * w0**2 / lambda_m
                z = np.linspace(0, z_max_mm*1e-3, 1000)
                w_z = w0 * np.sqrt(1 + (z/z_R)**2)
                divergence = np.degrees(np.arctan(lambda_m / (np.pi * w0)))
                computed_params = {"w0_um": w0_um,
                                  "rayleigh_range_mm": float(z_R*1e3),
                                  "divergence_deg": float(divergence),
                                  "beam_at_zmax_um": float(w_z[-1]*1e6)}
                st.subheader("Computed Parameters")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("Beam waist", f"{w0_um:.1f} um")
                with col2:
                    st.metric("Rayleigh range", f"{z_R*1e3:.2f} mm")
                with col3:
                    st.metric("Divergence", f"{divergence:.4f}°")
                with col4:
                    st.metric(f"Beam at {z_max_mm:.0f}mm", f"{w_z[-1]*1e6:.1f} um")
                fig, ax = plt.subplots(figsize=(8, 3))
                ax.plot(z*1e3, w_z*1e6, color="steelblue", linewidth=2)
                ax.plot(z*1e3, -w_z*1e6, color="steelblue", linewidth=2)
                ax.fill_between(z*1e3, -w_z*1e6, w_z*1e6, alpha=0.2, color="steelblue")
                ax.axvline(x=z_R*1e3, color="coral", linestyle="--", label="Rayleigh range")
                ax.set_xlabel("z (mm)")
                ax.set_ylabel("Beam radius (um)")
                ax.set_title("Gaussian beam propagation")
                ax.legend()
                ax.grid(True, alpha=0.3)
                st.pyplot(fig)

        elif category == "WDM System":
            st.subheader("WDM System Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                n_channels = st.number_input("Channels", value=8, min_value=1, max_value=96)
                spacing_nm = st.number_input("Spacing (nm)", value=0.8, format="%.2f")
            with col2:
                lambda_start = st.number_input("Start wavelength (nm)", value=1530.0, format="%.1f")
                fiber_loss = st.number_input("Fiber loss (dB/km)", value=0.2, format="%.2f")
            with col3:
                length_km = st.number_input("Link length (km)", value=100.0, format="%.1f")
                amp_spacing = st.number_input("Amp spacing (km)", value=80.0, format="%.1f")
            if st.button("Compute Parameters", type="secondary"):
                channels = [lambda_start + i*spacing_nm for i in range(n_channels)]
                n_amps = int(length_km / amp_spacing)
                computed_params = {
                    "n_channels": n_channels,
                    "span_loss_db": float(fiber_loss * amp_spacing),
                    "n_amplifiers": n_amps,
                    "total_capacity_gbps": float(n_channels * 100)
                }
                st.subheader("System Summary")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("Channels", n_channels)
                with col2:
                    st.metric("Span loss", f"{fiber_loss*amp_spacing:.1f} dB")
                with col3:
                    st.metric("Amplifiers", n_amps)
                with col4:
                    st.metric("Capacity", f"{n_channels*100} Gbps")
                st.subheader("Channel Wavelengths")
                cols = st.columns(min(8, n_channels))
                for i, lam in enumerate(channels[:8]):
                    with cols[i % 8]:
                        st.metric(f"Ch {i+1}", f"{lam:.1f} nm")

        elif category == "Resonator Stability":
            st.subheader("Resonator Parameters")
            col1, col2, col3 = st.columns(3)
            with col1:
                R1 = st.number_input("Mirror 1 radius R1 (m)", value=0.2, format="%.4f")
            with col2:
                R2 = st.number_input("Mirror 2 radius R2 (m)", value=0.2, format="%.4f")
            with col3:
                L  = st.number_input("Cavity length L (m)", value=0.1, format="%.4f")
            if st.button("Compute Parameters", type="secondary"):
                computed_params = check_resonator_stability(R1, R2, L)
                st.subheader("Stability Analysis")
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.metric("g1", f"{computed_params['g1']:.4f}")
                with col2:
                    st.metric("g2", f"{computed_params['g2']:.4f}")
                with col3:
                    st.metric("g1·g2", f"{computed_params['g1_g2']:.4f}")
                with col4:
                    st.metric("Status", computed_params['verdict'])
                if computed_params["stable"]:
                    st.success("Resonator is STABLE")
                else:
                    st.error("Resonator is UNSTABLE")
        
        else:
            st.info("Describe your problem above and click Get Design Recommendation.")
        
        st.markdown("---")
        if st.button("Get Design Recommendation", type="primary"):
            if not problem:
                st.warning("Please describe your design problem first.")
            else:
                with st.spinner("Analyzing your design..."):
                    try:
                        recommendation = get_design_recommendation(
                            problem, computed_params, category, api_key)
                        st.subheader("Design Recommendation")
                        st.markdown(recommendation)
                    except Exception as e:
                        st.error(f"Error: {e}")
'''

with open("/Users/mustafaabushagur/optical_design_advisor.py", "w") as f:
    f.write(app_code)

print("Complete Optical Design Advisor saved successfully")
print("Three modes available:")
print("  Standard Design  — component-level calculations")
print("  Agentic Agent    — autonomous design optimization")
print("  APRA             — full research assistant with reports")

Complete Optical Design Advisor saved successfully
Three modes available:
  Standard Design  — component-level calculations
  Agentic Agent    — autonomous design optimization
  APRA             — full research assistant with reports
